# CBUS mini-reproduction: Kaggle session 3 (Qwen2.5-3B, for criterion C5)
NEW notebook, GPU (T4 x2 is fine; only one GPU is used), Internet ON. Runs baseline and SPR with the small 3B model, same settings as the 7B runs (N=300, seed 0, max tokens 1000). About 30-40 min (my guess). Download `outputs.zip` at the end.

In [1]:
import os, subprocess, shutil, json, glob
REPO = 'https://github.com/sarrazer24/cbus-repro'
if not os.path.exists('/kaggle/working/cbus-repro'):
    subprocess.run(['git','clone',REPO,'/kaggle/working/cbus-repro'],check=True)
if not os.path.exists('/tmp/authors_data'):
    subprocess.run(['git','clone','--depth','1','https://github.com/iai-group/emnlp2026-simreading','/tmp/authors_data'],check=True)
os.chdir('/kaggle/working/cbus-repro')
os.environ['SIMREADING_DATA'] = '/tmp/authors_data'
subprocess.run('pip install -q -r requirements.txt',shell=True)
print(os.getcwd())

Cloning into '/kaggle/working/cbus-repro'...
Cloning into '/tmp/authors_data'...


/kaggle/working/cbus-repro


In [2]:
!nvidia-smi

Fri Nov 13 08:23:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [4]:
import time, requests
MODEL = 'Qwen/Qwen2.5-3B-Instruct'; TP = 1; NAME = 'local'
log = open('/tmp/vllm.log','w')
env = dict(os.environ, VLLM_ATTENTION_BACKEND='XFORMERS')
proc = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--served-model-name',NAME,
    '--dtype','half','--tensor-parallel-size',str(TP),'--max-model-len','8192','--gpu-memory-utilization','0.90',
    '--enable-prefix-caching','--port','8000'], stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models',timeout=2).ok: print('server ready after',i*5,'s'); break
    except Exception: pass
    if proc.poll() is not None: print('server exited; see /tmp/vllm.log'); break
    time.sleep(5)

server ready after 205 s


In [5]:
!tail -n 20 /tmp/vllm.log

(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/responses/{response_id}, Methods: GET
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/completions, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/messages, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /generative_scoring, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=103) INFO 11-13 08:32:00 [launcher.py:197] Route: /v1/messa

## Runs (explicit names so the 7B folders are never overwritten)

In [6]:
COMMON = '--base-url http://localhost:8000/v1 --model local --max-tokens 1000 --workers 32 --cache /tmp/cache.jsonl'
!python scripts/02_run_simulation.py --mode baseline --students 300 {COMMON} --name baseline-c4-qwen3b-n300-s0
!python scripts/02_run_simulation.py --mode spr --students 300 {COMMON} --name spr-c4-qwen3b-n300-s0
for f in sorted(glob.glob('runs/*qwen3b*/meta.json')):
    m=json.load(open(f)); print(m['name'],'| dropped_stage1',m['dropped_stage1_calls'],'| dropped_answers',m['dropped_answer_replies'],'| seconds',m['seconds'])

[run] baseline-c4-qwen3b-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=10388 dropped=8 (0.08%) usage={'calls': 10396, 'cache_hits': 0, 'input_tokens': 9440062, 'output_tokens': 71690} -> runs/baseline-c4-qwen3b-n300-s0
[run] spr-c4-qwen3b-n300-s0: 300 students x ~7 texts
  210/2100 student-texts
  420/2100 student-texts
  630/2100 student-texts
  840/2100 student-texts
  1050/2100 student-texts
  1260/2100 student-texts
  1470/2100 student-texts
  1680/2100 student-texts
  1890/2100 student-texts
  2100/2100 student-texts
[done] answered=9189 dropped=1207 (11.61%) usage={'calls': 11295, 'cache_hits': 0, 'input_tokens': 4541117, 'output_tokens': 275002} -> runs/spr-c4-qwen3b-n300-s0
baseline-c4-qwen3b-n300-s0 | dropped_stage1 0 | dr

In [7]:
!python scripts/03_evaluate.py --no-authors

 source                         name  n_students  mean_acc  score_sd   gap   jsd   ece  pearson  spearman  n_responses  dropped_pct
authors  Humans (test, ground truth)        1197     0.685     0.173   NaN   NaN   NaN      NaN       NaN          NaN          NaN
authors            Humans (held-out)        1161     0.661     0.181 0.026 0.006 0.027    0.895     0.845      35393.0          NaN
   mine    baseline-c4-local-n300-s0         300     0.897     0.056 0.210 0.451 0.214    0.267     0.219      10396.0        0.000
   mine   baseline-c4-qwen3b-n300-s0         300     0.822     0.068 0.135 0.281 0.141    0.310     0.233      10388.0        0.077
   mine       random-c4-none-n300-s0         300     0.366     0.089 0.321 0.627 0.316    0.354     0.319      10396.0        0.000
   mine         spr-c4-local-n300-s0         300     0.617     0.080 0.070 0.231 0.103    0.240     0.240      10218.0        1.712
   mine        spr-c4-qwen3b-n300-s0         300     0.561     0.090 0.125 0

In [8]:
shutil.make_archive('/kaggle/working/outputs','zip','.','runs')
proc.terminate()
for p in ['/tmp/cache.jsonl','/tmp/authors_data']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
print('exported -> /kaggle/working/outputs.zip; cache and authors data removed. Stop the session.')

exported -> /kaggle/working/outputs.zip; cache and authors data removed. Stop the session.
